In [0]:
from pyspark.sql.functions import (
    col, explode, split, trim, regexp_replace,
    when, lit, count, avg, sum as spark_sum,
    max as spark_max, min as spark_min,
    year, month, round as spark_round,
    desc, asc, countDistinct, stack
)
from pyspark.sql.types import IntegerType, FloatType, DoubleType

## 0. Chargement des tables depuis le Notebook 01

In [0]:
df_games      = spark.table("steam_games")
df_genres     = spark.table("steam_genres")
df_languages  = spark.table("steam_languages")
df_categories = spark.table("steam_categories")
df_platforms  = spark.table("steam_platforms")
df_publishers = spark.table("steam_publishers")

print("✅ Tables chargées et prêtes pour l'analyse")

✅ Tables chargées et prêtes pour l'analyse


## **PARTIE 1 — ANALYSE MACRO**

### 1.1. Quel publisher a sorti le plus de jeux sur Steam ?

In [0]:
display(
    df_games \
        .groupBy("publisher") \
        .count() \
        .orderBy("count", ascending=False) \
        .limit(20)
)

publisher,count
Big Fish Games,422
8floor,202
SEGA,165
Strategy First,151
Square Enix,141
Choice of Games,140
,132
HH-Games,132
Sekai Project,132
Ubisoft,127


Databricks visualization. Run in Databricks to view.

### 1.2 Quels sont les jeux les mieux notés ?

In [0]:
display(
    df_games
        .filter(col("total_reviews") >= 100)
        .select("name", "publisher", "positive", "negative", "total_reviews", "positive_ratio")
        .orderBy("positive_ratio", ascending=False)
        .limit(20)
)

name,publisher,positive,negative,total_reviews,positive_ratio
Dragon Spirits : Prologue,indienova,109,0,109,1.0
FIND ALL 2: Middle Ages,Very Very LITTLE Studio,132,0,132,1.0
未来战士,BIFROST CLOUD PTE. LTD.,116,0,116,1.0
秘封旅行 ~ Secret Sealing Travel,鸽屋谷,218,0,218,1.0
HAYAI,Chaoclypse,148,0,148,1.0
Yamafuda! 2nd station,KPC,109,0,109,1.0
Touhou Kaeizuka ～ Phantasmagoria of Flower View.,"Mediascape Co., Ltd.",119,0,119,1.0
The Void Rains Upon Her Heart,The Hidden Levels,496,0,496,1.0
祈風 Inorikaze,觀象草圖 Astrolabe Draft,327,0,327,1.0
Distant Memoraĵo,MangaGamer,114,0,114,1.0


Databricks visualization. Run in Databricks to view.

### 1.3 Y a-t-il des années avec plus de sorties ? 

In [0]:
display(
    df_games
        .filter(col("release_year").isNotNull())
        .groupBy("release_year")
        .count()
        .orderBy("release_year")
)

release_year,count
1997,2
1998,1
1999,3
2000,2
2001,4
2002,1
2003,3
2004,6
2005,6
2006,61


Databricks visualization. Run in Databricks to view.

### 1.4 Comment sont distribués les prix ?

In [0]:
# Jeux gratuits vs payants
display(
    df_games
        .withColumn("is_free",
            when(col("price") == 0, lit("Gratuit"))
            .otherwise(lit("Payant"))
        )
        .groupBy("is_free")
        .count()
        .orderBy("count", ascending=False)
)

is_free,count
Payant,47911
Gratuit,7779


Databricks visualization. Run in Databricks to view.

In [0]:
# Distribution des prix pour les jeux payants
display(
    df_games
        .filter(col("price") > 0)
        .select("price")
        .groupBy("price")
        .count()
        .orderBy("price")
)

price,count
28.0,20
29.0,11
30.0,2
31.0,6
37.0,19
38.0,7
39.0,7
41.0,10
44.0,1
45.0,2


Databricks visualization. Run in Databricks to view.

In [0]:
# Y a-t-il beaucoup de jeux avec une remise ?
display(
    df_games
        .withColumn("has_discount",
            when(col("discount") > 0, lit("Avec remise"))
            .otherwise(lit("Sans remise"))
        )
        .groupBy("has_discount")
        .count()
)

has_discount,count
Sans remise,53172
Avec remise,2518


Databricks visualization. Run in Databricks to view.

### 1.5 Quelles sont les langues les plus représentées ?

In [0]:
display(
    df_languages
        .groupBy("language")
        .count()
        .orderBy("count", ascending=False)
        .limit(20)
)

language,count
English,55116
German,14019
French,13426
Russian,12922
Simplified Chinese,12782
Spanish - Spain,12233
Japanese,10368
Italian,9304
Portuguese - Brazil,6750
Korean,6600


Databricks visualization. Run in Databricks to view.

### 1.6 Combien de jeux sont interdits aux moins de 16 / 18 ans ?

In [0]:
display(
    df_games
        .withColumn("age_category",
            when(col("required_age") >= 18, lit("18+"))
            .when(col("required_age") >= 16, lit("16+"))
            .when(col("required_age") >= 12, lit("12+"))
            .when(col("required_age") > 0,   lit("Autre restriction"))
            .otherwise(lit("Tous publics"))
        )
        .groupBy("age_category")
        .count()
        .orderBy("count", ascending=False)
)

age_category,count
Tous publics,55029
12+,333
18+,230
16+,76
Autre restriction,22


Databricks visualization. Run in Databricks to view.

## **PARTIE 2 — ANALYSE GENRES**

### 2.1 Quels sont les genres les plus représentés ?

In [0]:
display(
    df_genres
        .groupBy("genre")
        .count()
        .orderBy("count", ascending=False)
        .limit(20)
)

genre,count
Indie,39681
Action,23759
Casual,22086
Adventure,21431
Strategy,10895
Simulation,10836
RPG,9534
Early Access,6145
Free to Play,3393
Sports,2666


Databricks visualization. Run in Databricks to view.

### 2.2 Quels genres ont le meilleur ratio de reviews positives ?

In [0]:
display(
    df_genres
        .filter(col("total_reviews") >= 100)
        .groupBy("genre")
        .agg(
            count("appid").alias("nb_jeux"),
            spark_round(avg("positive_ratio"), 3).alias("avg_positive_ratio"),
            spark_sum("total_reviews").alias("total_reviews")
        )
        .orderBy("avg_positive_ratio", ascending=False)
        .limit(20)
)

genre,nb_jeux,avg_positive_ratio,total_reviews
Game Development,32,0.826,28137
Photo Editing,20,0.826,590014
Web Publishing,34,0.819,37946
Animation & Modeling,85,0.81,711283
Design & Illustration,92,0.808,693687
Sexual Content,20,0.798,6958
Casual,4735,0.797,11217985
Adventure,6635,0.793,35004692
Indie,10607,0.791,36133776
Utilities,155,0.783,771700


Databricks visualization. Run in Databricks to view.

### 2.3 Est-ce que certains publishers ont des genres favoris ?

In [0]:
# Top 10 publishers par nombre de jeux
top_publishers = (df_games 
    .groupBy("publisher") 
    .count() 
    .orderBy("count", ascending=False) 
    .limit(10) 
    .select("publisher")
)

# Genres favoris de ces top publishers
display(
    df_genres
        .join(top_publishers, on="publisher", how="inner")
        .groupBy("publisher", "genre")
        .count()
        .orderBy("publisher", "count", ascending=False)
)

publisher,genre,count
Ubisoft,Action,70
Ubisoft,Adventure,45
Ubisoft,Strategy,22
Ubisoft,RPG,19
Ubisoft,Simulation,18
Ubisoft,Racing,14
Ubisoft,Casual,11
Ubisoft,Indie,7
Ubisoft,Free to Play,6
Ubisoft,Sports,5


Databricks visualization. Run in Databricks to view.

### 2.4 Quels sont les genres les plus lucratifs ?

In [0]:
display(
    df_genres
        .groupBy("genre")
        .agg(
            count("appid").alias("nb_jeux"),
            spark_round(avg("price"), 2).alias("avg_price"),
            spark_round(avg("positive_ratio"), 3).alias("avg_positive_ratio")
        )
        .orderBy("avg_price", ascending=False)
        .limit(20)
)

genre,nb_jeux,avg_price,avg_positive_ratio
Web Publishing,89,2179.63,0.72
Game Development,159,2128.22,0.738
Photo Editing,105,2032.62,0.703
Audio Production,195,1964.5,0.675
Design & Illustration,406,1905.84,0.721
Software Training,164,1903.27,0.7
Video Production,247,1895.89,0.68
Animation & Modeling,322,1877.63,0.707
Accounting,16,1443.25,0.723
Education,317,1434.17,0.685


Databricks visualization. Run in Databricks to view.

## **PARTIE 3 — ANALYSE PLATEFORMES**

### 3.1 La majorité des jeux sont-ils sur Windows / Mac / Linux ?

In [0]:
total = df_games.count()

display(
    df_games
        .agg(
            spark_round(spark_sum(col("windows").cast(IntegerType())) / total * 100, 2).alias("Windows"),
            spark_round(spark_sum(col("mac").cast(IntegerType())) / total * 100, 2).alias("Mac"),
            spark_round(spark_sum(col("linux").cast(IntegerType())) / total * 100, 2).alias("Linux")
        )
        .select(
            stack(
                lit(3),
                lit("Windows"), col("Windows"),
                lit("Mac"),     col("Mac"),
                lit("Linux"),   col("Linux")
            ).alias("platform", "percentage")
        )
)

platform,percentage
Windows,99.97
Mac,22.93
Linux,15.19


Databricks visualization. Run in Databricks to view.

### 3.2 Certains genres sont-ils préférentiellement disponibles sur certaines plateformes ?

In [0]:
display(
    df_games
        .select(
            explode(col("genres_list")).alias("genre"),
            col("windows"), col("mac"), col("linux")
        )
        .withColumn("genre", trim(col("genre")))
        .filter(col("genre") != "")
        .groupBy("genre")
        .agg(
            spark_round(avg(col("windows").cast(DoubleType())) * 100, 1).alias("Windows"),
            spark_round(avg(col("mac").cast(DoubleType())) * 100, 1).alias("Mac"),
            spark_round(avg(col("linux").cast(DoubleType())) * 100, 1).alias("Linux"),
            count("genre").alias("nb_jeux")
        )
        .orderBy("nb_jeux", ascending=False)
        .limit(20)
        .select(
            col("genre"),
            col("nb_jeux"),
            stack(
                lit(3),
                lit("Windows"), col("Windows"),
                lit("Mac"),     col("Mac"),
                lit("Linux"),   col("Linux")
            ).alias("platform", "percentage")
        )
)

genre,nb_jeux,platform,percentage
Indie,39681,Windows,100.0
Action,23759,Windows,100.0
Casual,22086,Windows,100.0
Adventure,21431,Windows,100.0
Strategy,10895,Windows,100.0
Simulation,10836,Windows,100.0
RPG,9534,Windows,100.0
Early Access,6145,Windows,100.0
Free to Play,3393,Windows,99.9
Sports,2666,Windows,100.0


Databricks visualization. Run in Databricks to view.

## **PARTIE 4 — ANALYSES COMPLÉMENTAIRES**

### 4.1. Quels genres sont en train d'émerger ces dernières années ?


In [0]:
display(
    df_genres
        .filter(col("release_year").isNotNull())
        .filter(col("release_year") >= 2015)
        .groupBy("release_year", "genre")
        .count()
        .orderBy("release_year", "count", ascending=False)
)

release_year,genre,count
2022,Indie,5327
2022,Action,3184
2022,Casual,3126
2022,Adventure,3125
2022,Strategy,1561
2022,Simulation,1528
2022,RPG,1471
2022,Early Access,1226
2022,Sports,333
2022,Racing,322


Databricks visualization. Run in Databricks to view.

### 4.2 Corrélation prix vs reviews positives

In [0]:
display(
    df_games
        .filter(col("total_reviews") >= 100)
        .filter(col("price") > 0)
        .withColumn("price_range",
            when(col("price") < 5,   lit("< 5$"))
            .when(col("price") < 10,  lit("5$ - 10$"))
            .when(col("price") < 20,  lit("10$ - 20$"))
            .when(col("price") < 30,  lit("20$ - 30$"))
            .when(col("price") < 60,  lit("30$ - 60$"))
            .otherwise(lit("> 60$"))
        )
        .groupBy("price_range")
        .agg(
            count("appid").alias("nb_jeux"),
            spark_round(avg("positive_ratio"), 3).alias("avg_positive_ratio")
        )
        .orderBy("price_range")
)

price_range,nb_jeux,avg_positive_ratio
20$ - 30$,7,0.756
30$ - 60$,143,0.717
> 60$,13206,0.792


Databricks visualization. Run in Databricks to view.

### 4.3 Top jeux par nombre de propriétaires

In [0]:
display(
    df_games
        .select("name", "publisher", "owners_range", "owners_lower", "positive_ratio", "price")
        .orderBy("owners_lower", ascending=False)
        .limit(20)
)

name,publisher,owners_range,owners_lower,positive_ratio,price
Dota 2,Valve,"200,000,000 .. 500,000,000",200000000,0.828414231133127,0.0
Counter-Strike: Global Offensive,Valve,"50,000,000 .. 100,000,000",50000000,0.8830547135268165,0.0
Team Fortress 2,Valve,"50,000,000 .. 100,000,000",50000000,0.9364670347299824,0.0
New World,Amazon Games,"50,000,000 .. 100,000,000",50000000,0.6878634560073899,1999.0
PUBG: BATTLEGROUNDS,"KRAFTON, Inc.","50,000,000 .. 100,000,000",50000000,0.5661084992616564,0.0
War Thunder,Gaijin Distribution KFT,"20,000,000 .. 50,000,000",20000000,0.7843758695938722,0.0
Path of Exile,Grinding Gear Games,"20,000,000 .. 50,000,000",20000000,0.8713598904660896,0.0
Unturned,Smartly Dressed Games,"20,000,000 .. 50,000,000",20000000,0.914741508681185,0.0
Fall Guys: Ultimate Knockout,Ravenscourt,"20,000,000 .. 50,000,000",20000000,0.8128310433402526,0.0
Half-Life 2: Lost Coast,Valve,"20,000,000 .. 50,000,000",20000000,0.8825862387013467,0.0


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

### 4.4 Les jeux multi-plateformes sont-ils plus populaires ?

In [0]:
display(
    df_games
        .withColumn("nb_platforms",
            col("windows").cast(IntegerType()) +
            col("mac").cast(IntegerType()) +
            col("linux").cast(IntegerType())
        )
        .groupBy("nb_platforms")
        .agg(
            count("appid").alias("nb_jeux"),
            spark_round(avg("positive_ratio"), 3).alias("avg_positive_ratio"),
            spark_round(avg("price"), 2).alias("avg_price")
        )
        .orderBy("nb_platforms")
)

nb_platforms,nb_jeux,avg_positive_ratio,avg_price
1,41285,0.722,776.77
2,7599,0.773,721.82
3,6806,0.783,809.7


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    df_games
        .withColumn("platforms_combo",
            when(
                col("windows") & ~col("mac") & ~col("linux"),
                lit("Windows only")
            ).when(
                col("windows") & col("mac") & ~col("linux"),
                lit("Windows + Mac")
            ).when(
                col("windows") & ~col("mac") & col("linux"),
                lit("Windows + Linux")
            ).when(
                col("windows") & col("mac") & col("linux"),
                lit("Windows + Mac + Linux")
            ).when(
                ~col("windows") & col("mac") & col("linux"),
                lit("Mac + Linux")
            ).otherwise(lit("Autre"))
        )
        .groupBy("platforms_combo")
        .agg(
            count("appid").alias("nb_jeux"),
            spark_round(avg("positive_ratio"), 3).alias("avg_positive_ratio"),
            spark_round(avg("price"), 2).alias("avg_price")
        )
        .orderBy("nb_jeux", ascending=False)
)

platforms_combo,nb_jeux,avg_positive_ratio,avg_price
Windows only,41271,0.722,776.26
Windows + Mac + Linux,6806,0.783,809.7
Windows + Mac,5951,0.77,740.51
Windows + Linux,1647,0.782,654.41
Autre,14,0.513,2292.36
Mac + Linux,1,0.841,499.0


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.